# Challenge 2 - ANOVA

In statistics, **Analysis of Variance (ANOVA)** is also used to analyze the differences among group means. The difference between t-test and ANOVA is the former is ued to compare two groups whereas the latter is used to compare three or more groups. [Read more about the difference between t-test and ANOVA](http://b.link/anova24).

From the ANOVA test, you receive two numbers. The first number is called the **F-value** which indicates whether your null-hypothesis can be rejected. The critical F-value that rejects the null-hypothesis varies according to the number of total subjects and the number of subject groups in your experiment. In [this table](http://b.link/eda14) you can find the critical values of the F distribution. **If you are confused by the massive F-distribution table, don't worry. Skip F-value for now and study it at a later time. In this challenge you only need to look at the p-value.**

The p-value is another number yielded by ANOVA which already takes the number of total subjects and the number of experiment groups into consideration. **Typically if your p-value is less than 0.05, you can declare the null-hypothesis is rejected.**

In this challenge, we want to understand whether there are significant differences among various types of pokemons' `Total` value, i.e. Grass vs Poison vs Fire vs Dragon... There are many types of pokemons which makes it a perfect use case for ANOVA.

In [28]:
# Import libraries
import pandas as pd
import scipy.stats as stats


In [2]:
# Load the data:
df = pd.read_csv('/Users/anna/iron_hack/lab-hypothesis-testing-2/your-code/Pokemon.csv')

In [3]:
df.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


**To achieve our goal, we use three steps:**

1. **Extract the unique values of the pokemon types.**

1. **Select dataframes for each unique pokemon type.**

1. **Conduct ANOVA analysis across the pokemon types.**

#### First let's obtain the unique values of the pokemon types. These values should be extracted from Type 1 and Type 2 aggregated. Assign the unique values to a variable called `unique_types`.

*Hint: the correct number of unique types is 19 including `NaN`. You can disregard `NaN` in next step.*

In [12]:
# Your code here

# Obtain unique values from Type 1 and Type 2
type_1_unique = df['Type 1'].unique()
type_2_unique = df['Type 2'].unique()

# Combine unique types and remove NaN values
unique_types = pd.Series(list(set(type_1_unique) | set(type_2_unique))).tolist()
len(unique_types) # you should see 19

19

In [23]:
unique_types

['Poison',
 'Ghost',
 'Electric',
 'Water',
 'Fighting',
 'Flying',
 'Fire',
 'Normal',
 'Psychic',
 'Bug',
 'Grass',
 'Rock',
 'Dark',
 'Steel',
 'Ice',
 'Fairy',
 'Dragon',
 'Ground',
 nan]

#### Second we will create a list named `pokemon_totals` to contain the `Total` values of each unique type of pokemons.

Why we use a list instead of a dictionary to store the pokemon `Total`? It's because ANOVA only tells us whether there is a significant difference of the group means but does not tell which group(s) are significantly different. Therefore, we don't need know which `Total` belongs to which pokemon type.

*Hints:*

* Loop through `unique_types` and append the selected type's `Total` to `pokemon_groups`.
* Skip the `NaN` value in `unique_types`. `NaN` is a `float` variable which you can find out by using `type()`. The valid pokemon type values are all of the `str` type.
* At the end, the length of your `pokemon_totals` should be 18.

In [25]:
pokemon_totals = []

# Loop through unique types and collect Total values
for pokemon_type in unique_types:
    if isinstance(pokemon_type, str): 
        # Append the Total values for Type 1
        total_values_type_1 = df.loc[df['Type 1'] == pokemon_type, 'Total'].tolist()    
        total_values_type_2 = df.loc[df['Type 2'] == pokemon_type, 'Total'].tolist()

        pokemon_totals.append(total_values_type_1+total_values_type_2)

# Check the length of pokemon_totals
print(len(pokemon_totals))  # Should output 18

18


In [26]:
pokemon_totals

[[288,
  438,
  275,
  365,
  505,
  273,
  365,
  505,
  245,
  455,
  325,
  500,
  340,
  490,
  535,
  302,
  467,
  458,
  329,
  479,
  330,
  500,
  300,
  490,
  329,
  474,
  320,
  494,
  318,
  405,
  525,
  625,
  195,
  205,
  395,
  495,
  320,
  395,
  490,
  305,
  450,
  300,
  390,
  490,
  335,
  515,
  310,
  405,
  500,
  600,
  250,
  390,
  430,
  385,
  400,
  280,
  515,
  260,
  360,
  485,
  294,
  464],
 [310,
  405,
  500,
  600,
  435,
  295,
  455,
  555,
  295,
  455,
  348,
  498,
  495,
  485,
  525,
  680,
  680,
  303,
  483,
  275,
  370,
  520,
  309,
  474,
  335,
  335,
  335,
  335,
  494,
  494,
  494,
  494,
  236,
  380,
  480,
  480,
  440,
  335,
  480,
  303,
  483,
  325,
  448,
  520,
  520,
  600],
 [320,
  485,
  325,
  465,
  330,
  480,
  490,
  525,
  580,
  205,
  280,
  365,
  510,
  610,
  360,
  580,
  295,
  475,
  575,
  405,
  405,
  263,
  363,
  523,
  405,
  535,
  540,
  440,
  520,
  520,
  520,
  520,
  520,
  295,
  49

#### Now we run ANOVA test on `pokemon_totals`.

*Hints:*

* To conduct ANOVA, you can use `scipy.stats.f_oneway()`. Here's the [reference](http://b.link/scipy44).

* What if `f_oneway` throws an error because it does not accept `pokemon_totals` as a list? The trick is to add a `*` in front of `pokemon_totals`, e.g. `stats.f_oneway(*pokemon_groups)`. This trick breaks the list and supplies each list item as a parameter for `f_oneway`.

In [32]:
# Your code here
# Conducting the ANOVA test
try:
    f_statistic, p_value = stats.f_oneway(*pokemon_totals)
    print(f"F-statistic: {f_statistic}, P-value: {p_value}")
    
    # Interpretation of the results
    alpha = 0.05  # significance level
    if p_value < alpha:
        print("Reject the null hypothesis: There is a significant difference between the group means.")
    else:
        print("Fail to reject the null hypothesis: No significant difference between the group means.")
        
except Exception as e:
    print(f"An error occurred: {e}")


F-statistic: 6.6175382960055344, P-value: 2.6457458815984803e-15
Reject the null hypothesis: There is a significant difference between the group means.


#### Interpret the ANOVA test result. Is the difference significant?

# Your comment here
There is a significant difference between the group means.